In [ ]:
import torch   
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.amp import autocast
 


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"


In [ ]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import random_split, DataLoader, Dataset

In [ ]:
def normalization(x,mini , maxi):
    return (x - mini) / (maxi - mini)

def denormalization(x ,mini , maxi):
    return x * (maxi - mini) + mini

In [ ]:
print(device)

cuda


In [ ]:
class SpectrogramDataset(Dataset):
    def __init__(self, spec_path, minim, maxim):
        self.file_paths = [os.path.join(spec_path, f)
                           for f in os.listdir(spec_path) if f.endswith('.npy')]
        self.minim = minim
        self.maxim = maxim

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        spec = np.load(self.file_paths[idx])
        spec_tensor = torch.from_numpy(spec).float()
        spec_norm = (spec_tensor - self.minim) / (self.maxim - self.minim)
        return spec_norm

def get_data(spec_path, minim, maxim, batch_size=12, split_ratio=0.8):
    dataset = SpectrogramDataset(spec_path, minim, maxim)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader



In [ ]:
import numpy as np
import os

def get_global_min_max(spec_path):
    global_min = np.inf
    global_max = -np.inf

    for fname in os.listdir(spec_path):
        if fname.endswith(".npy"):
            data = np.load(os.path.join(spec_path, fname))

            data_min = data.min()
            data_max = data.max()

            if data_min < global_min:
                global_min = data_min
            if data_max > global_max:
                global_max = data_max

    return global_min, global_max




In [ ]:
minim, maxim = get_global_min_max("C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\specc")
        
print(maxim)
print(minim)

train_loader, test_loader = get_data("C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\spec", minim, maxim, batch_size=12, split_ratio=0.8)



53.74848
-42.518272


In [ ]:
class Cond_VAE(nn.Module): # marche pas ouf
    def __init__(self, latent_dim=400):
        super(Cond_VAE, self).__init__()
        
        # Encoder - réduit progressivement la dimension
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # 513 x 1000
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),  # 257 x 500
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 129 x 250
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 65 x 125
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # 33 x 63
            nn.ReLU()
        )
        
        # Taille du feature map après encodage
        self.feature_size = 33 * 63 * 256
        
        # Couches pour calculer moyenne et logvar
        self.fc_mu = nn.Linear(self.feature_size, latent_dim)
        self.fc_logvar = nn.Linear(self.feature_size, latent_dim)
        
        # Première couche du décodeur
        self.decoder_input = nn.Linear(latent_dim, self.feature_size)
        
        # Decoder - augmente progressivement la dimension
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),  # 65 x 125
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # 129 x 250
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # 257 x 500
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # 513 x 1000
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # 1025 x 2000
            nn.Upsample(size=(1025, 2000), mode='bilinear', align_corners=False),
            nn.Sigmoid() 
        )
        
        self.latent_dim = latent_dim
        
    def encode(self, x):
        # x est de forme [batch_size, 1, 1025, 2000]
        x= x.unsqueeze(1)  # Ajouter la dimension du canal
        x = self.encoder(x)
        x = x.view(x.size(0), -1)  # Aplatir pour les couches linéaires
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        # z est de forme [batch_size, latent_dim]
        z = self.decoder_input(z)
        z = z.view(z.size(0), 256, 33, 63)  # Redimensionner pour convolution
        z = self.decoder(z)
        return z
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar
    
    def sample(self, num_samples):
        with torch.no_grad():
            z = torch.randn(num_samples, self.latent_dim, device=device)
            samples = self.decode(z)
            return samples
    
    def reconstruct(self, x):
        with torch.no_grad():
            mu, _ = self.encode(x)
            return self.decode(mu)
    

In [ ]:
def train_model_vae(data_loader,test_loader, model, criterion, optimizer, nepochs, device, minim ,maxim, scheduler=None):

    scaler = GradScaler(device='cuda')
    loss_affichage = []
    max = np.inf
    loss_test = []
    recon_loss_plot = []
    kl_loss_plot = []
    mu_plot = []
    logvar_plot = []
    logvar_std_plot = []
    mu_std_plot = []
    for epoch in range(nepochs):
        train_loss = 0
        model.train()
        for batch_idx, input_ in enumerate(tqdm(data_loader, desc=f"Epoch {epoch}")):
            input_ = input_.to(device)  # shape: (batch, 1025, 2000)

            # Découpage en petits morceaux

            optimizer.zero_grad()
            with autocast('cuda'):  # 👈 Ici, on fait tout en float16 sauf exceptions

                recon, mu, logvar = model(input_)

                recon = recon.squeeze(1)  # Enlever la dimension du canal
                loss , recon_loss, kl_loss = criterion(recon, input_, mu ,logvar,epoch)

            scaler.scale(loss).backward()

            # 💥 Clip des gradients pour éviter explosion
            scaler.unscale_(optimizer)  # Obligatoire avant clip avec AMP
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            recon_loss_plot.append(recon_loss.detach().cpu().item())
            kl_loss_plot.append(kl_loss.detach().cpu().item())
            mu_plot.append(mu.mean().detach().cpu().item())
            logvar_plot.append(logvar.mean().detach().cpu().item())
            logvar_std_plot.append(logvar.std().detach().cpu().item())
            mu_std_plot.append(mu.std().detach().cpu().item())
            """loss.backward()
            optimizer.step()"""

            train_loss += loss.detach().cpu().item()
            
            if max > loss.item():
                max = loss.item() 
                torch.save(model.state_dict(), "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\VAE\\modele\\best_modele.pth")
            '''if batch_idx % 40 == 0:
                print(f"Batch {batch_idx}: Loss = {loss.item():.6f}")'''
        avg_loss = train_loss / len(data_loader)
        loss_affichage.append(avg_loss)	
        print(f"Epoch {epoch}: Training loss = {avg_loss:.6f}")
        del input_, recon, mu, logvar, loss, recon_loss, kl_loss  # Clean all tensors
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

        if scheduler is not None:
            scheduler.step(avg_loss)
        
        if epoch % 10 == 0:
            test_loss = 0
            model.eval()
            
            with torch.no_grad():
                for batch_idx, input_ in enumerate(test_loader):
                    input_ = input_.to(device)
                    recon, mu, logvar = model(input_)
                    loss, recon_loss, kl_loss = criterion(recon, input_ ,mu , logvar,epoch)
                    test_loss += loss.detach().cpu().item()
                loss_test.append(test_loss/len(test_loader))
                print(f"Test loss = {test_loss/len(test_loader):.6f}")
            testeur(data_loader, model, device , epoch, loss_test , loss_affichage ,recon_loss_plot , kl_loss_plot, mu_plot , logvar_plot,  logvar_std_plot, mu_std_plot ,minim , maxim)  
            
        del input_, recon, mu, logvar, loss, recon_loss, kl_loss  # Clean all tensors
        torch.cuda.empty_cache()


In [ ]:
def testeur(data_loader, model, device , epoch , loss_test, loss_affichage, recon_loss_plot , kl_loss_plot, mu_plot,logvar_plot , logvar_std_plot, mu_std_plot , minim, maxim):
    path = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\projet deep L3S2\\VAE"
    
    
    plt.figure(figsize=(10, 4))
    plt.plot(kl_loss_plot, label='KL Divergence Loss', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('KL Divergence Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"loss_reconstruction_kl_{epoch}epochs.png"))
    plt.close()
    
    plt.figure(figsize=(10, 4))
    plt.plot(recon_loss_plot, label='Reconstruction Loss', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('KL Divergence Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"loss_reconstruction_{epoch}epochs.png"))
    plt.close()
    
    plt.figure(figsize=(10, 4))
    plt.plot(mu_plot, label='mu', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('mu Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"mu_moy{epoch}epochs.png"))
    plt.close()
    
    plt.figure(figsize=(10, 4))
    plt.plot(logvar_plot, label='logvar', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('logvar Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"logvar_moy{epoch}epochs.png"))
    plt.close()
    
    plt.figure(figsize=(10, 4))
    plt.plot(logvar_std_plot, label='logvar', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('logvar_std Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"logvar_std{epoch}epochs.png"))
    plt.close()
    
    plt.figure(figsize=(10, 4))
    plt.plot(mu_std_plot, label='logvar', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('mu_std Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"mu_std{epoch}epochs.png"))
    plt.close()
    
    original = data_loader.dataset[8]
    original = original.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Reconstruction
        original_input = original.permute(0, 2, 1)  # (1, 2000, 1025)
        reconstructed, mu, logvar = model(original_input)  # 👈 changement ici
        reconstructed = reconstructed.permute(0, 2, 1) 

    original = original.squeeze(0).cpu().detach().numpy()
    reconstructed = reconstructed.squeeze(0).cpu().detach().numpy()


    original = denormalization(original, minim, maxim)

    reconstructed2 = denormalization(reconstructed, minim, maxim)
        
        
    def save_audio_original(path, spec, epoch):
        # Enregistrer le fichier audio
        audio_path = os.path.join(path + "\\son", f"audio_original.wav")
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512)
        sf.write(audio_path, signal, 22050)
         
    
    def save_audio_reconstructed(path, spec, epoch):
        # Enregistrer le fichier audio
        audio_path = os.path.join(path + "\\son", f"audio_reconstruit{epoch}.wav")
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512)

        sf.write(audio_path, signal, 22050)

    def save_image_original(path, image, epoch):
        # Enregistrer l'image
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(image, y_axis='log', x_axis='time')
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"{image} : {epoch}")
        img_path = os.path.join(path + "\\images" , "images_original.png")
        plt.savefig(img_path)
        plt.close()

    def save_image_reconstruit(path, image, epoch):
        # Enregistrer l'image
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(image, y_axis='log', x_axis='time')
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"{image} : {epoch}")
        img_path = os.path.join(path + "\\images", f"images_reconstuites{epoch}.png")
        plt.savefig(img_path)
        plt.close()
    
    save_audio_original(path, original, epoch)
    save_audio_reconstructed(path, reconstructed2, epoch)
    save_image_original(path, original, epoch)
    save_image_reconstruit(path, reconstructed2, epoch)
    
    if epoch > 0 : 
        plt.figure(figsize=(10, 4))
        train = list(range(len(loss_affichage)))         # [0, 1, 2, ..., 49]
        test = list(range(0, epoch+1 , 10))   # [0, 10, 20, 30, 40]
   
        # 2. Les courbes
        plt.plot(train, loss_affichage, label='Train Loss', color='blue')
        plt.plot(test, loss_test, label='Test Loss', color='orange', marker='o')

        # 3. Habillage
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Train vs Test Loss')
        plt.legend()
        plt.grid(True)

        # 4. Sauvegarde
        plt.savefig(os.path.join(path + "\\loss", f"loss_comparaison_{epoch}epochs.png"))
        plt.close()

    
    if epoch > 30 : 
        loss__test_tronque = loss_test[1:]
        loss_affichage_tronque = loss_affichage[10:]
        plt.figure(figsize=(10, 4))
        train = list(range(len(loss_affichage_tronque)))         # [0, 1, 2, ..., 49]
        test = list(range(0, epoch, 10))   # [0, 10, 20, 30, 40]

        # 2. Les courbes
        plt.plot(train, loss_affichage_tronque, label='Train Loss', color='blue')
        plt.plot(test, loss__test_tronque, label='Test Loss', color='orange', marker='o')

        # 3. Habillage
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Train vs Test Loss')
        plt.legend()
        plt.grid(True)

        # 4. Sauvegarde
        plt.savefig(os.path.join(path + "\\loss", f"loss_comparaison_tronque.png"))
        plt.close()
        
    
    

In [ ]:
def vae_loss_fn(recon, x, mu, logvar,epoch):
    # Stabilisation avec log1p
    recon = torch.clamp(recon, min=-0.99)
    x = torch.clamp(x, min=-0.99)
    recon = torch.log1p(recon)
    x = torch.log1p(x)
    beta = min(1.0, epoch / 50)  # ou un autre planning

    # Reconstruction loss
    recon_loss = F.mse_loss(recon, x, reduction='mean')

    # KL divergence
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss
    total_loss = recon_loss + beta*kl_loss
    return total_loss , recon_loss, kl_loss


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
#criterion = auraloss.freq.MultiResolutionSTFTLoss() # a tester e profondeur
#criterion = nn.MSELoss()  #avant
from torch.optim.lr_scheduler import ReduceLROnPlateau


criterion = vae_loss_fn # Utiliser la classe personnalisée
model = Cond_VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005) # 0.0011 bien pour spec

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=4, verbose=True)
"""scheduler = StepLR(optimizer, step_size=20, gamma=0.3)  """


train_loader, test_loader = get_data("C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\spec", minim, maxim, batch_size=2, split_ratio=0.8)


train_model_vae(train_loader, test_loader, model, criterion, optimizer, minim = minim , maxim = maxim, nepochs=101, device=device, scheduler=scheduler)


KeyboardInterrupt: 

In [ ]:
def test_best_model(model_path, data_loader, minim, maxim, device, epoch):
    path = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\projet deep L3S2\\VAE"
    model = Cond_VAE().to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()

    # Prend un échantillon du dataset
    original = data_loader.dataset[8]
    original = original.unsqueeze(0).to(device)  # [1, 1025, 2000]
    original_input = original.permute(0, 2, 1)  # [1, 2000, 1025]

    with torch.no_grad():
        reconstructed, mu, logvar = model(original_input)


    original = original.squeeze(0).cpu().numpy()
    reconstructed = reconstructed.squeeze(0).cpu().numpy()

    original = denormalization(original, minim, maxim)
    reconstructed = denormalization(reconstructed, minim, maxim)

    def save_audio(path, spec, filename):
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512)
        sf.write(os.path.join(path + "\\son", filename), signal, 22050)

    def save_image(path, spec, filename):
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(spec, y_axis='log', x_axis='time')
        plt.colorbar(format='%+2.0f dB')
        plt.title(filename)
        plt.savefig(os.path.join(path + "\\images", filename))
        plt.close()

    save_audio(path, original, "audio_original.wav")
    save_audio(path, reconstructed, f"audio_reconstruit_{epoch}.wav")
    save_image(path, original, "images_original.png")
    save_image(path, reconstructed, f"images_reconstuites_{epoch}.png")

    print(f"✅ Test terminé pour epoch {epoch}, fichiers sauvegardés.")
    
test_best_model("C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\projet deep L3S2\\VAE\\modele\\best_modele.pth", test_loader, minim, maxim, device, 0)

c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


LibsndfileError: Error opening 'C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\projet deep L3S2\\VAE\\son\\audio_reconstruit_0.wav': Format not recognised.